In [60]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [61]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"

as_of = datetime.date(2026, 1, 7)
start = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 0, 0))
end = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 23, 59))

# from SDRUtils.data.builder import SDRDataBuilder
# sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
# df = sdr.grab_sdr_trades(
# 	start_timestamp=start,
# 	end_timestamp=end,
# 	agency="CFTC",
# 	asset_class="RATES",
# )
# df

In [62]:
# from SDRUtils.products.usd.sofr_swaps import USD_SOFR_SwapProduct 
# USD_SOFR_SwapProduct().build_classification_dataframe(start=start, end=end, cache_path=cache_path)

from SDRUtils.products.usd.usd_swaptions import USD_Swaptions
sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path, ignore_cache=True)

Classifying Trades: 100%|██████████| 530/530 [00:00<00:00, 1540.09trade/s]


In [69]:
# temp = sdf.copy()
# temp["execution_timestamp"] = temp["execution_timestamp"].astype(str)
# temp.to_excel("temp.xlsx",index=False)
# temp[temp["package_type"] == "SWAPTION"].to_excel("filter_swaptions_trades_no_pkg.xlsx")
# sdf["trade_label"].value_counts()

sdf["package_type"].value_counts()
# sdf[~sdf["trade_label"].str.contains("4Dx10Y")]["package_type"].value_counts()
# sdf[(sdf["package_type"] == "STRADDLE") & (sdf["trade_label"].str.contains("1Mx2Y"))]
# sdf[(sdf["trade_label"].str.contains("1Yx10Y"))]
# sdf[sdf["trade_label"].str.contains("3Mx10Y")]

package_type
SWAPTION               215
STRADDLE               126
VERTICAL_SPREAD_1x1     19
VERTICAL_SPREAD_1x3      3
RISK_REVERSAL            3
Name: count, dtype: int64

In [67]:
sdf.loc[244:246]

,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,is_notional_capped,...,upi_underlier_name,unique_product_identifier,platform_identifier,cleared,package_indicator,package_transaction_price,option_premium_amount,package_confidence,package_reason,package_legs_count
244,NEWT-TRAD,1652241161000002301 / 1652400689000001301,2026-01-07 14:30:56+00:00,2026-01-07,2027-04-28,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1Y CONSTANT FOMC_2027042...,50000000.0,USD,False,...,NA/Swap OIS USD,QZJ92TTHTSF0,BILT,N,False,NaN,"292,500 / 0",0.8,platform=BILT; time_delta_max=0.0s; premium_mo...,2.0
245,CORR-TRAD,1652186256000000101 / 1652186257000000201,2026-01-07 14:34:07+00:00,2026-01-07,2026-04-07,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 3Mx10Y PAYER...,50000000.0,USD,False,...,NA/Swap OIS USD,QZZGWPNBF5R3 / QZNLQ8T0N0SX,BGCD,N,True,"1,120,000",0,1.0,platform=BGCD; time_delta_max=0.0s; premium_mo...,2.0
246,NEWT-TRAD,1652200981000000101 / 1652200982000000201,2026-01-07 14:36:55+00:00,2026-01-07,2027-01-07,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y PAYER...,25000000.0,USD,False,...,NA/Swap Fxd Flt USD,QZWXKVHB5F8V / QZMMWR8JKZQ8,BGCD,N,True,"1,242,500","1,242,500",1.0,platform=BGCD; time_delta_max=0.0s; premium_mo...,2.0


In [34]:
# ["platform_identifier"].value_counts()
# sdf[sdf["package_type"] == "RISK_REVERSAL"]
# sdf = sdf[~(sdf["trade_label"].str.contains("4Dx10Y")) & (sdf["package_type"] == "SWAPTION") & ~(sdf["platform_identifier"].isin(["BILT", "XXXX"]))]
# sdf["execution_timestamp"] = sdf["execution_timestamp"].astype(str)
# sdf.to_excel("temp1.xlsx")

# sdf[(sdf["package_type"] == "STRADDLE") & (sdf[ "package_indicator"] == False)]
# sdf["package_type"].value_counts()
# sdf[(sdf["package_type"] == "VERTICAL_SPREAD_1x1")]

In [68]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from SDRUtils.products._swaptions.pricer import usd_swaption_straddle_pricer_from_row, usd_swaption_leg_pricer_from_row

mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))

# usd_swaption_leg_pricer_from_row(sdf.loc[491], pricer, leg="payer", fwd_prem=248750)
# usd_swaption_leg_pricer_from_row(sdf.loc[15], pricer, leg="receiver", fwd_prem=565000.0)
usd_swaption_straddle_pricer_from_row(sdf.loc[245], pricer), usd_swaption_straddle_pricer_from_row(sdf.loc[246], pricer)

(USDSwaptionStraddlePricerResult(ql_payer_swaption=<QuantLib.QuantLib.Swaption; proxy of <Swig Object of type 'ext::shared_ptr< Swaption > *' at 0x0000029302119C50> >, ql_receiver_swaption=<QuantLib.QuantLib.Swaption; proxy of <Swig Object of type 'ext::shared_ptr< Swaption > *' at 0x00000293091D8930> >, notional=50000000.0, fwd_prem=1120000.0, bpvol_yr=67.43210741486095, dv01=1195.5424893417912, gamma01=413.7852780016524, vega01=16550.506382640186, theta1d=-6217.510600067908),
 USDSwaptionStraddlePricerResult(ql_payer_swaption=<QuantLib.QuantLib.Swaption; proxy of <Swig Object of type 'ext::shared_ptr< Swaption > *' at 0x000002930553C240> >, ql_receiver_swaption=<QuantLib.QuantLib.Swaption; proxy of <Swig Object of type 'ext::shared_ptr< Swaption > *' at 0x000002930553EB50> >, notional=25000000.0, fwd_prem=1242500.0, bpvol_yr=74.58600205244409, dv01=149.55808755237558, gamma01=164.80838659064202, vega01=16206.521891670303, theta1d=-1656.9981988304062))

USDSwaptionStraddlePricerResult(ql_payer_swaption=<QuantLib.QuantLib.Swaption; proxy of <Swig Object of type 'ext::shared_ptr< Swaption > *' at 0x000001BD04075350> >, ql_receiver_swaption=<QuantLib.QuantLib.Swaption; proxy of <Swig Object of type 'ext::shared_ptr< Swaption > *' at 0x000001BD02675CB0> >, notional=230000000.0, fwd_prem=13305500.0, bpvol_yr=80.15796368212092, dv01=-104.90472914712154, gamma01=421.9206590175954, vega01=146076.87368989503, theta1d=-4007.949709035456)

In [45]:
136695.32935272245 * 2
# + 146076.87368989503

273390.6587054449

In [56]:
# sdf.loc[336]["package_reason"]
sdf.loc[492]

event_action                                                          NEWT-TRAD
trade_id                              1693390051000000201 / 1693444086000002901
execution_timestamp                                   2026-01-12 06:46:25+00:00
effective_date                                              2026-01-12 00:00:00
expiration_date                                             2028-01-12 00:00:00
product_type                                 SWAPTION_PAYER / SWAPTION_RECEIVER
trade_label                   USD-SOFR-COMPOUND 1D CONSTANT 2Yx10Y PAYER EUR...
notional                                                            100000000.0
notional_currency                                                           CNY
is_notional_capped                                                        False
estimated_pv01                                                              0.0
package_type                                                           STRADDLE
package_id                              

In [46]:
4.09* np.sqrt(252)

np.float64(64.92673717352505)

In [51]:
ids = [
1696946219000001501,
1696946220000001601,
1696946221000001701,
1696949372000000201,


]

# df[df["Dissemination Identifier"].isin([str(id) for id in ids])].to_csv("temp_trades.csv",index=False)

sdf[sdf["trade_id"].isin([str(id) for id in ids])].to_dict(orient="records")

# df[df["Dissemination Identifier"].isin([str(id) for id in ids])].to_dict(orient="records")
# df["Action type"].value_counts()
# df["Event type"].value_counts()

[{'event_action': 'NEWT-TRAD',
  'trade_id': '1696946220000001601',
  'execution_timestamp': Timestamp('2026-01-12 16:10:46+0000', tz='UTC'),
  'effective_date': Timestamp('2026-01-12 00:00:00'),
  'expiration_date': Timestamp('2026-02-12 00:00:00'),
  'product_type': 'SWAPTION_RECEIVER',
  'trade_label': 'USD-SOFR-COMPOUND 1D CONSTANT 1Mx20Y RECEIVER EURO VANILLA PHYS',
  'notional': 25000000.0,
  'notional_currency': 'USD',
  'is_notional_capped': False,
  'estimated_pv01': 0.0,
  'package_type': 'SWAPTION',
  'package_id': None,
  'package_legs': None,
  'underlying_expiration_date': Timestamp('2046-02-17 00:00:00'),
  'tenor_years': 20.027397260273972,
  'tenor_label': '20Y',
  'forward_start_years': 0.08493150684931507,
  'forward_label': '1M',
  'premium': 248750.0,
  'exercise_style': 'EUROPEAN',
  'strike': 0.0417,
  'upi_underlier_name': 'NA/Swap Fxd Flt USD',
  'unique_product_identifier': 'QZXZSN00ZVCG',
  'platform_identifier': 'BILT',
  'cleared': 'N',
  'package_indicator